# 🏎️ Sentinel F1: Real-Time Adaptive Anomaly Detection Pipeline
### Challenge 2 — Sentinel Track | Numenta Anomaly Benchmark (NAB) & 2026 F1 Telemetry
**Team Apex Signal**

This notebook demonstrates the end-to-end streaming anomaly detection architecture for **Challenge 2 (Sentinel)**:
1. High-frequency streaming telemetry ingestion (1-second intervals)
2. Sliding-window buffer with feature embedding ($W=8$, $\mu$, $\sigma$, $\Delta x$, lags)
3. All **8 Permitted Algorithms** (Rolling Z-score, Isolation Forest, LOF, One-Class SVM, Autoencoders, Temporal Clustering, Adaptive Thresholding, and Hybrid Consensus)
4. **Covariate-Shift Adaptive Thresholding** $\tau_t = \mu_W + k\cdot\sigma_W \cdot (1 + \Delta_{\text{regime}})$
5. Benchmark evaluation across Numenta Anomaly Benchmark (NAB) standard datasets
6. Exporting compliant prediction files matching the organizer starter kit schema.

In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print('Environment initialized successfully.')

## 1. Load Data Streams (F1 Telemetry & NAB Benchmarks)
We ingest 1-second high-frequency racing telemetry alongside standard Numenta Anomaly Benchmark files (`ambient_temperature_system_failure.csv`, `nyc_taxi.csv`, `art_daily_flatmiddle.csv`, `cpu_utilization_asg_misconfiguration.csv`).

In [2]:
# Load pre-generated evaluation results
f1_df = pd.read_csv('../results/predictions_f1_telemetry_2026.csv')
nab_df = pd.read_csv('../results/predictions_nab_benchmark.csv')
metrics_df = pd.read_csv('../results/metrics_benchmark_summary.csv')

print(f'F1 Telemetry: {len(f1_df)} records loaded.')
print(f'NAB Benchmark Suite: {len(nab_df)} records loaded across datasets:')
print(nab_df['dataset'].value_counts())

## 2. Permitted Algorithms Evaluation Matrix
Comparison of all 8 permitted algorithms under the challenge specifications:

In [3]:
metrics_df.sort_values(by='event_f1', ascending=False)

## 3. Streaming Telemetry & Adaptive Threshold Visualization
Below we visualize the telemetry stream with the dynamic threshold $\tau_t$ adapting to covariate shifts (e.g. pit stops and Safety Car periods).

In [4]:
plt.figure(figsize=(14, 5))
plt.plot(f1_df['value'], label='Lap Time / Telemetry Value', color='#00e5ff', lw=1.5)
anom = f1_df[f1_df['label'] == 'ANOMALOUS']
plt.scatter(anom.index, anom['value'], color='#e10600', s=45, label='Flagged Anomaly', zorder=5)
plt.title('Sentinel F1 Real-Time Streaming Anomaly Detection', fontsize=14, fontweight='bold')
plt.xlabel('Streaming Window ID (1-Second Windows)', fontsize=11)
plt.ylabel('Pace (Seconds)', fontsize=11)
plt.grid(True, alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Multi-Head Anomaly Score Distribution
Notice that when an anomaly occurs, the multi-head consensus score $S_t$ strictly clears the local threshold $\tau_t$.

In [5]:
plt.figure(figsize=(14, 4))
plt.plot(f1_df['anomaly_score'], label='Anomaly Score (S_t)', color='#e10600', lw=2)
plt.axhline(0.613, color='#00e5ff', linestyle='--', label='Baseline Threshold (k=2.45σ)')
plt.title('Anomaly Score Horizon vs. Adaptive Decision Frontier', fontsize=13, fontweight='bold')
plt.xlabel('Window Index', fontsize=11)
plt.ylabel('Normalized Score [0, 1]', fontsize=11)
plt.grid(True, alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Output Verification & Schema Validation
Verify that the exported predictions file matches the exact schema requested in the organizer starter kit:
`timestamp, value, anomaly_score, label, window_id, detection_delay_ms, interpretable_evidence`

In [6]:
print('Sample Predictions File Rows:')
print(f1_df.head(10).to_string(index=False))

print('\nAnomaly Class Distribution in F1 Stream:')
print(f1_df['label'].value_counts())